# Data analysis for ingest pipeline

Validate that TasteMender's imported audio-feature data is complete, internally consistent, and faithful to the original AcousticBrainz source data.

This work should identify any issues introduced during ingestion, transformation, normalization, or storage, and establish whether each feature provides enough useful variation to support recommendation ranking.

In [ ]:
import os, uuid, sys, random
import django
import numpy as np
import numpy.typing as npt
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Enable access to Django project inside current notebook
PROJECT_PATH = os.path.dirname(os.path.abspath(""))
sys.path.append(PROJECT_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "music_recommendation.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from recommend_api.models import GenreDortmund, GenreRosamerica, Track, Artist, Album

### Display genre distribution charts for 2 classifiers: Dortmund and Rosamerica

In [ ]:
feature_store = np.load("../features_and_index.npz", allow_pickle=True, mmap_mode="r")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Pull genre codes for all tracks into a pandas Series
dort = pd.Series(feature_store["genre_dortmund"], name="count")
rosa = pd.Series(feature_store["genre_rosamerica"])

# Count and plot Dortmund, genre labels are stored in Django models
counts_dort = dort.value_counts()
dort_labels_by_code = dict(GenreDortmund.objects.values_list("code", "label"))
labels_dort = [dort_labels_by_code[code] for code in counts_dort.index]

axes[0].pie(counts_dort, labels=labels_dort, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Genre distribution (Dortmund)")
axes[0].legend(labels_dort, loc="best", fontsize="small")

# Count and plot Rosamerica, genre labels are stored in Django models
counts_rosa = rosa.value_counts()
rosa_labels_by_code = dict(GenreRosamerica.objects.values_list("code", "label"))
labels_rosa = [
    rosa_labels_by_code[code] for code in counts_rosa.index
]
axes[1].pie(counts_rosa, labels=labels_rosa, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Genre distribution (Rosamerica)")
axes[1].legend(labels_rosa, loc="best", fontsize="small")

plt.tight_layout()
plt.show()

The Dortmund classification designates a whopping 77.8% as being electronic music while Rosamerica splits up genres more evenly. The official AcousticBrainz [accuracy metrics](https://acousticbrainz.org/datasets/accuracy) put Dortmund at 60.25% and Rosamerica at 87.56%.

We ca safely conclude that the Rosamerica classification is the best choice for determining musical genre in TasteMender due to its higher accuracy.

### Feature Distribution - Normalized vs Raw
Audio features go through 3 different representations before they can be used by the recommender logic. They are:
- **raw** - the original audio features that are extracted from AcousticBrainz JSON dataset
- **standardized** - each feature is transformed to have mean 0 and standard deviation 1 using `StandardScaler`
- **normalized** - the complete feature vector for each track is divided by its Euclidean norm for cosine similarity

It's currently unclear if and how changing representations affects the recommendation logic. Since this is the core of the project, it worthwhile to make a deep dive into this aspect as well as the logic in general. We'll begin by comparing the representations for all of the audio features.

Questions that should be answered:
- How are features distributed in general?
- Does standardization and normalization change the shape of the distribution?

In [ ]:
features_raw = feature_store["feature_matrix_raw"]
features_norm = feature_store["feature_matrix"]
feature_names = feature_store["feature_names"]

print("Audio features: \n", feature_names)
print("Shape of raw feature matrix: ", features_raw.shape)
print("Shape of normalized feature matrix:", features_norm.shape)

In [ ]:
# Generate standardized matrix from raw features
scaler = StandardScaler(with_mean=True, with_std=True).fit(features_raw)
features_scaled = scaler.transform(features_raw).astype(np.float32)

# Re-generate the normalized matrix so we can compare against the one saved in the store
temp_features_norm = features_scaled / (
    np.linalg.norm(features_scaled, axis=1, keepdims=True) + 1e-8
)

# Check normalized matrix in store is correct
assert np.array_equal(temp_features_norm, features_norm) is True

In [ ]:
feature_df = pd.DataFrame(
    features_raw,
    columns=feature_names,
)

feature_df.head()

In [ ]:
basic_stats = feature_df.agg(
    [
        "count",
        "mean",
        "std",
        "min",
        "max",
    ]
).T
basic_stats

In [ ]:
percentiles = feature_df.quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

percentiles

#### Danceability

In [ ]:
danc_raw = features_raw[:,feature_names == 'danceability'].flatten()
dance_scaled = features_scaled[:,feature_names == 'danceability'].flatten()
danc_normalized = features_norm[:,feature_names == 'danceability'].flatten()

plt.hist(danc_raw, bins=200, density=True, histtype="step", label="Raw")
plt.hist(dance_scaled, bins=200, density=True, histtype="step", label="Scaled")
plt.hist(danc_normalized, bins=200, density=True, histtype="step", label="Normalized")

plt.legend()
plt.show()

In [ ]:
pd.DataFrame({
    "range": [
        [danc_raw.min(), danc_raw.max()],
        [dance_scaled.min(), dance_scaled.max()],
        [danc_normalized.min(), danc_normalized.max()],
    ],
    "mean": [
        danc_raw.mean(),
        dance_scaled.mean(),
        danc_normalized.mean(),
    ],
    "std": [
        danc_raw.std(),
        dance_scaled.std(),
        danc_normalized.std(),
    ]

}, index=["raw", "scaled", "normalized"])

In [ ]:
near_zero = (danc_raw < 0.01).mean() * 100
near_one = (danc_raw > 0.99).mean() * 100
uncertain = ((danc_raw > 0.4) & (danc_raw < 0.6)).mean() * 100

print(f"Below 0.01: {near_zero:.2f}%")
print(f"Above 0.99: {near_one:.2f}%")
print(f"Uncertain predictions: {uncertain:.2f}%")

##### Conclusions
Values for the danceability feature are heavily concentrated at the extremes, ~40% of them being close to either zero or one. Clearly they were generated by a classifier that was determining if a song was danceable or not. The feature doesn't show _how_ danceable the song is.

This brings into question how we should view the features. So far we've assumed they were continuous variables that mapped to some sort of distribution (normal distribution). But the findings show that they're clustered at the extremes.

### Missing Data

#### Missing Release Year

In [ ]:
years: npt.NDArray[np.int16] = np.array(feature_store["years"])
num_tracks = len(years)
zero_years = np.sum(years == 0)

print(f"Tracks missing a release year: {zero_years:,} / {num_tracks:,}")

mbids_raw: npt.NDArray[np.void] = feature_store["mbids"]
zero_year_mbids_raw = random.sample(mbids_raw[years == 0].tolist(), 10)
zero_year_mbids = [str(uuid.UUID(bytes=mbid)) for mbid in zero_year_mbids_raw]

zero_year_tracks_qs = (
    Track.objects.filter(musicbrainz_recordingid__in=zero_year_mbids)
    .select_related("album")
    .prefetch_related("artists")
)

zero_year_tracks_df = pd.DataFrame(
    [
        (
            track.musicbrainz_recordingid,
            track.title,
            ", ".join(artist.name for artist in track.artists.all()),
            track.album.name if track.album else "<no album>",
            track.album.date if track.album else None,
        )
        for track in zero_year_tracks_qs
    ],
    columns=["mbid", "title", "artists", "album", "album_date"],
)

print("Sample of tracks missing a release year:")
display(zero_year_tracks_df)